In [1]:
## Basics Deep Agents
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

## StateBackend

In [3]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

agent = create_deep_agent(model = "groq:openai/gpt-oss-120b")

agent2 = create_deep_agent(
   model = "groq:openai/gpt-oss-120b",
   backend = StateBackend(),
)

In [4]:
## Invoking the agent

result = agent2.invoke({
    "messages" : [{
        "role":"user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video \n2. Edit video \n3. upload video\n"
            "Then tell me you've done it."
         )
    }]
})

print("\n ----- Agent Reply ----")
print(result["messages"][-1].content)


 ----- Agent Reply ----
Done.


In [5]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video \n2. Edit video \n3. upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='a37eb2f0-4c1f-43d4-ad34-0576559b294e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants to create a file at /notes/todo.txt with exact content lines. Need to write file. Then respond that done. No need for todo list. Just write file. Use write_file tool.', 'tool_calls': [{'id': 'fc_74d4154b-fd73-475b-a25d-fe5bd8e881cc', 'function': {'arguments': '{"content":"1. Record video \\n2. Edit video \\n3. upload video","file_path":"/notes/todo.txt"}', 'name': 'write_file'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 5482, 'total_tokens': 5578, 'completion_time': 0.201813113, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.317378695, 'prompt_tokens_details': None, '

In [7]:
print("\n --- Backend Check ----")
files = result.get("files",{})

if files:
    print(f"StateBackend is working - {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n {path}\n{'-' * 40 }\n {content}")

else:
    print("No files found in state. either the agent did not write a file,"
    "or the backend isn't wired up correctly.")


 --- Backend Check ----
StateBackend is working - 1 file(s) in state:

 /notes/todo.txt
----------------------------------------
 {'content': '1. Record video \n2. Edit video \n3. upload video', 'encoding': 'utf-8', 'created_at': '2026-07-14T14:32:11.711200+00:00', 'modified_at': '2026-07-14T14:32:11.711200+00:00'}


In [9]:
followup = agent2.invoke({
    "messages" : result["messages"] + [{
        "role" :"user",
        "content" : "Read /notes/todo.txt back to me ."
    }],
    "files":result.get("files",{}),
})

print("\n -- Read Back (same thread) ---")
print(followup["messages"][-1].content)


 -- Read Back (same thread) ---
1. Record video 
2. Edit video 
3. upload video


## FilEeSystem Backend(local Disk)

In [13]:
from deepagents.backends import FilesystemBackend
from langchain_core.messages.content import create_text_block
ROOT = "."

agent = create_deep_agent(model = "groq:openai/gpt-oss-120b",backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))
print(f"Agent created by FilesystemBackend(root_dir={ROOT!r}),")
print("  Files written by the agent will appear on your actual disk.")

Agent created by FilesystemBackend(root_dir='.'),
  Files written by the agent will appear on your actual disk.


In [14]:
## invoke agent

result = agent.invoke({
     "messages" : [{
        "role":"user",
        "content":(
            "Create a file at /notes/todo.txt with exactly this content. \n"
            "1. Record video \n2.Edit video \n3.Upload video\n"
            "Then Tell me you ve done it"
        )
     }]
})

print("\n---Agenty Reply ------")
print(result["messages"][-1].content)


---Agenty Reply ------
Done.


## Stores Backend


In [15]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()


agent = create_deep_agent(
    model = "groq:openai/gpt-oss-120b",
    backend = StoreBackend(
        namespace=lambda rt: ("demo-user",),
    ),
    store = store
)

print("Agent created with store backend + static namespace.")

Agent created with store backend + static namespace.
